# Distributed Multi-Rank Stencil & Asynchronous Halo Exchange — MPI + CUDA

**Individual laboratory · 4 MPI ranks · one NVIDIA T4 · 512×512 global field · 400 steps**

GitHub handle for this notebook: **`allfocusSS`**. This notebook starts with real environment and GPU UUID verification; it does not contain invented execution measurements. The six answers at the end are filled from the actual `submission_manifest.json` using the supplied finalizer **after** your real Colab run.

Run on **Google Colab → Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all**. Keep the original execution logs. See `README.md` for the final GitHub submission steps.

## Cell 1 — Environment verification and MPI installation

In [1]:
!nvidia-smi --query-gpu=name,uuid,compute_cap,memory.total --format=csv
!command -v mpirun >/dev/null || (apt-get update -qq && apt-get install -y -qq openmpi-bin libopenmpi-dev)
!mpirun --version | head -n 2
!pip install -q mpi4py numba numpy


name, uuid, compute_cap, memory.total [MiB]
Tesla T4, GPU-561dd3dd-1373-c02a-71f0-8c2c965c50bc, 7.5, 15360 MiB
mpirun (Open MPI) 4.1.6



## Cell 2 — Student identity (your real GitHub account)

In [2]:
GITHUB_USERNAME = "allfocusSS"
assert GITHUB_USERNAME.strip() and GITHUB_USERNAME != "REPLACE_WITH_YOUR_GITHUB_USERNAME"
print(f"Student GitHub handle: {GITHUB_USERNAME}")

Student GitHub handle: allfocusSS


## Cell 3 — Implemented five-task MPI/CUDA solver

In [3]:
%%writefile distributed_cuda_solver.py
"""Four-rank MPI + CUDA reaction-diffusion stencil laboratory.

Run in a Google Colab NVIDIA T4 runtime:
  mpirun --allow-run-as-root --oversubscribe -np 4 \
      python3 distributed_cuda_solver.py <YOUR_REAL_GITHUB_USERNAME>

Every process owns only a 258 x 258 padded subdomain; field arrays are never
collected to a root process. Requires mpi4py, NumPy, and numba.cuda.
"""

from __future__ import annotations

import datetime as dt
import hashlib
import json
import math
import re
import subprocess
import sys
from dataclasses import dataclass
from pathlib import Path

import numpy as np
from mpi4py import MPI
from numba import cuda, float64

TILE_DIM = 16
SHARED_DIM = 18
GLOBAL_NX = 512
GLOBAL_NY = 512
STEPS = 400
LAMBDA_DAMP = 0.15
UP, DOWN, LEFT, RIGHT = range(4)
SEND_TAGS = (120, 121, 122, 123)
OPPOSITE = (DOWN, UP, RIGHT, LEFT)
DIRECTIONS = ("up", "down", "left", "right")


def get_gpu_uuid() -> str:
    """Read the *actual* active GPU hardware identifier, never a placeholder."""
    value = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=uuid", "--format=csv,noheader"],
        encoding="utf-8",
        timeout=20,
    ).strip().splitlines()[0].strip()
    if not value or not value.startswith("GPU-"):
        raise RuntimeError(f"Unexpected nvidia-smi GPU UUID: {value!r}")
    return value


def derive_student_seed(github_user: str, gpu_uuid: str) -> int:
    """Match the SHA-256 username::GPU-UUID seed rule in the PDF starter."""
    digest = hashlib.sha256(
        f"{github_user.strip()}::{gpu_uuid}".encode("utf-8")
    ).hexdigest()
    return int(digest[:16], 16)


@dataclass
class DistributedContext:
    cart: object
    rank: int
    size: int
    coords: tuple[int, int]
    neighbors: tuple[int, int, int, int]
    seed: int
    gpu_uuid: str
    github_user: str
    ny: int
    nx: int


def init_distributed_context(github_user: str) -> DistributedContext:
    comm = MPI.COMM_WORLD
    size = comm.Get_size()
    if size != 4:
        raise RuntimeError(f"Expected exactly four MPI ranks (2 x 2); got {size}")
    if not github_user.strip() or github_user.startswith("REPLACE_"):
        raise ValueError("Provide your actual GitHub handle as argument 1")
    cuda.select_device(0)
    cart = comm.Create_cart(dims=[2, 2], periods=[False, False], reorder=False)
    rank = cart.Get_rank()
    coords = tuple(int(v) for v in cart.Get_coords(rank))
    up, down = cart.Shift(0, 1)
    left, right = cart.Shift(1, 1)
    uuid = get_gpu_uuid() if rank == 0 else None
    uuid = cart.bcast(uuid, root=0)
    seed = derive_student_seed(github_user, uuid)
    return DistributedContext(
        cart, rank, size, coords, (up, down, left, right),
        seed, uuid, github_user, GLOBAL_NY // 2, GLOBAL_NX // 2,
    )


def initialize_subdomains(ctx: DistributedContext):
    """Construct seeded physics using rank-local, including ghost, coordinates."""
    dx = 1.0 / (GLOBAL_NX + 1)
    dy = 1.0 / (GLOBAL_NY + 1)
    row_rank, col_rank = ctx.coords
    # Local padded index 0 maps to the preceding global boundary/neighbour.
    # Owned index 1 maps to the rank's first global interior cell.
    x = (col_rank * ctx.nx + np.arange(ctx.nx + 2)) * dx
    y = (row_rank * ctx.ny + np.arange(ctx.ny + 2)) * dy
    xx, yy = np.meshgrid(x, y, indexing="xy")

    kappa = 0.08 + 0.16 * (
        np.sin(2 * np.pi * xx) ** 2 * np.cos(2 * np.pi * yy) ** 2
    )
    rng = np.random.default_rng(ctx.seed)
    centers = rng.uniform(0.28, 0.72, size=(2, 2))
    amplitudes = rng.uniform(0.7, 1.1, size=2)
    widths = (0.075, 0.09)

    u = np.zeros((ctx.ny + 2, ctx.nx + 2), dtype=np.float64)
    for (xc, yc), amp, width in zip(centers, amplitudes, widths):
        u[1:-1, 1:-1] += amp * np.exp(
            -((xx[1:-1, 1:-1] - xc) ** 2 +
              (yy[1:-1, 1:-1] - yc) ** 2) / (2.0 * width**2)
        )

    # Positive, compact forcing; cubic damping dominates the global integral
    # over the short 400-step experiment. Exact energy behavior is still
    # measured and reported rather than assumed.
    source = np.zeros_like(u)
    source[1:-1, 1:-1] = 0.01 * np.exp(
        -((xx[1:-1, 1:-1] - 0.5) ** 2 +
          (yy[1:-1, 1:-1] - 0.5) ** 2) / (2.0 * 0.035**2)
    )
    return u, np.ascontiguousarray(kappa), source, dx, dy


# TASK 3: each CUDA lane packs four independent boundary-vector elements.
@cuda.jit
def pack_halos_kernel(u, send_up, send_down, send_left, send_right,
                      ny, nx):
    t = cuda.grid(1)
    if t < nx:
        send_up[t] = u[1, t + 1]
        send_down[t] = u[ny, t + 1]
    if t < ny:
        send_left[t] = u[t + 1, 1]
        send_right[t] = u[t + 1, nx]


@cuda.jit
def unpack_halos_kernel(u, recv_up, recv_down, recv_left, recv_right,
                        ny, nx, has_up, has_down, has_left, has_right):
    t = cuda.grid(1)
    if t < nx:
        u[0, t + 1] = recv_up[t] if has_up else 0.0
        u[ny + 1, t + 1] = recv_down[t] if has_down else 0.0
    if t < ny:
        u[t + 1, 0] = recv_left[t] if has_left else 0.0
        u[t + 1, nx + 1] = recv_right[t] if has_right else 0.0


# TASK 4: 16x16 CUDA threads, precisely 18x18 shared-memory tile.
@cuda.jit
def tiled_stencil_kernel(u, kappa, source, u_next, dt_step, inv_dx2,
                         inv_dy2, r_start, r_end, c_start, c_end):
    tx = cuda.threadIdx.x
    ty = cuda.threadIdx.y
    r = r_start + cuda.blockIdx.y * TILE_DIM + ty
    c = c_start + cuda.blockIdx.x * TILE_DIM + tx
    last_r = u.shape[0] - 1
    last_c = u.shape[1] - 1

    rr = r
    cc = c
    if rr > last_r:
        rr = last_r
    if cc > last_c:
        cc = last_c

    smem = cuda.shared.array(shape=(SHARED_DIM, SHARED_DIM), dtype=float64)
    smem[ty + 1, tx + 1] = u[rr, cc]
    if tx == 0:
        cleft = c - 1
        if cleft < 0:
            cleft = 0
        smem[ty + 1, 0] = u[rr, cleft]
    if tx == TILE_DIM - 1:
        cright = c + 1
        if cright > last_c:
            cright = last_c
        smem[ty + 1, SHARED_DIM - 1] = u[rr, cright]
    if ty == 0:
        rtop = r - 1
        if rtop < 0:
            rtop = 0
        smem[0, tx + 1] = u[rtop, cc]
    if ty == TILE_DIM - 1:
        rbottom = r + 1
        if rbottom > last_r:
            rbottom = last_r
        smem[SHARED_DIM - 1, tx + 1] = u[rbottom, cc]
    cuda.syncthreads()  # all threads, even the inactive edge threads

    if r < r_end and c < c_end:
        center = smem[ty + 1, tx + 1]
        north = smem[ty, tx + 1]
        south = smem[ty + 2, tx + 1]
        west = smem[ty + 1, tx]
        east = smem[ty + 1, tx + 2]
        kc = kappa[r, c]
        kn = 0.5 * (kc + kappa[r - 1, c])
        ks = 0.5 * (kc + kappa[r + 1, c])
        kw = 0.5 * (kc + kappa[r, c - 1])
        ke = 0.5 * (kc + kappa[r, c + 1])
        flux_x = (ke * (east - center) - kw * (center - west)) * inv_dx2
        flux_y = (ks * (south - center) - kn * (center - north)) * inv_dy2
        u_next[r, c] = center + dt_step * (
            flux_x + flux_y - LAMBDA_DAMP * center**3 + source[r, c]
        )


# Device-only blockwise reduction, no global-array gathering onto rank zero.
@cuda.jit
def diagnostics_kernel(u, prev, part_energy, part_residual, ny, nx):
    tid = cuda.threadIdx.x
    idx = cuda.blockIdx.x * cuda.blockDim.x + tid
    scratch_energy = cuda.shared.array(shape=256, dtype=float64)
    scratch_residual = cuda.shared.array(shape=256, dtype=float64)
    energy = 0.0
    residual_sq = 0.0
    if idx < ny * nx:
        r = idx // nx + 1
        c = idx % nx + 1
        energy = u[r, c]
        difference = u[r, c] - prev[r, c]
        residual_sq = difference * difference
    scratch_energy[tid] = energy
    scratch_residual[tid] = residual_sq
    cuda.syncthreads()
    stride = 128
    while stride >= 1:
        if tid < stride:
            scratch_energy[tid] += scratch_energy[tid + stride]
            scratch_residual[tid] += scratch_residual[tid + stride]
        cuda.syncthreads()
        stride //= 2
    if tid == 0:
        part_energy[cuda.blockIdx.x] = scratch_energy[0]
        part_residual[cuda.blockIdx.x] = scratch_residual[0]


class HaloBuffers:
    def __init__(self, ny: int, nx: int):
        self.send_device = []
        self.recv_device = []
        self.send_host = []
        self.recv_host = []
        for n in (nx, nx, ny, ny):
            self.send_device.append(cuda.device_array(n, dtype=np.float64))
            self.recv_device.append(cuda.to_device(np.zeros(n, dtype=np.float64)))
            self.send_host.append(cuda.pinned_array(n, dtype=np.float64))
            self.recv_host.append(cuda.pinned_array(n, dtype=np.float64))


def exchange_halos_async(ctx: DistributedContext, buffers: HaloBuffers):
    """Receive opposite-direction tags and send the local-direction tags."""
    requests = []
    for d, nbr in enumerate(ctx.neighbors):
        if nbr != MPI.PROC_NULL:
            requests.append(ctx.cart.Irecv(
                buffers.recv_host[d], source=nbr, tag=SEND_TAGS[OPPOSITE[d]]
            ))
    for d, nbr in enumerate(ctx.neighbors):
        if nbr != MPI.PROC_NULL:
            requests.append(ctx.cart.Isend(
                buffers.send_host[d], dest=nbr, tag=SEND_TAGS[d]
            ))
    return requests


def launch_box(u, kappa, source, u_next, dt_step, dx, dy,
               r0, r1, c0, c1, stream):
    if r0 >= r1 or c0 >= c1:
        return
    grid = (
        ((c1 - c0 + TILE_DIM - 1) // TILE_DIM),
        ((r1 - r0 + TILE_DIM - 1) // TILE_DIM),
    )
    tiled_stencil_kernel[grid, (TILE_DIM, TILE_DIM), stream](
        u, kappa, source, u_next, dt_step,
        1.0 / (dx * dx), 1.0 / (dy * dy), r0, r1, c0, c1,
    )


def step_overlapped(ctx, u, u_next, kappa, source, dt_step, dx, dy,
                    halo, stream_boundary, stream_interior,
                    overlap: bool, sample_events: bool = False):
    """A(pack/D2H) -> B(MPI+interior) -> C(H2D/unpack+boundary).

    Serialized baseline deliberately waits for MPI completion before
    launching the same interior tiled kernel. Mathematics/order are identical.
    """
    n = max(ctx.nx, ctx.ny)
    pack_halos_kernel[(n + 255) // 256, 256, stream_boundary](
        u, *halo.send_device, ctx.ny, ctx.nx,
    )
    for d, nbr in enumerate(ctx.neighbors):
        if nbr != MPI.PROC_NULL:
            halo.send_device[d].copy_to_host(
                halo.send_host[d], stream=stream_boundary
            )
    stream_boundary.synchronize()  # MPI may not touch a D2H buffer until ready
    requests = exchange_halos_async(ctx, halo)

    events = None
    if sample_events:
        events = [cuda.event(timing=True) for _ in range(4)]
    if not overlap:
        MPI.Request.Waitall(requests)

    if events is not None:
        events[0].record(stream_interior)
    launch_box(u, kappa, source, u_next, dt_step, dx, dy,
               2, ctx.ny, 2, ctx.nx, stream_interior)
    if events is not None:
        events[1].record(stream_interior)
    if overlap:
        MPI.Request.Waitall(requests)
    else:
        # The baseline does not overlap GPU interior work with H2D/ring work.
        stream_interior.synchronize()

    # Interprocess writes are now finished. Stage received buffers on the GPU.
    for d, nbr in enumerate(ctx.neighbors):
        if nbr != MPI.PROC_NULL:
            halo.recv_device[d].copy_to_device(
                halo.recv_host[d], stream=stream_boundary
            )
    flags = tuple(int(nbr != MPI.PROC_NULL) for nbr in ctx.neighbors)
    unpack_halos_kernel[(n + 255) // 256, 256, stream_boundary](
        u, *halo.recv_device, ctx.ny, ctx.nx, *flags,
    )

    if events is not None:
        events[2].record(stream_boundary)
    # Four disjoint pieces of the one-cell perimeter ring.
    launch_box(u, kappa, source, u_next, dt_step, dx, dy,
               1, 2, 1, ctx.nx + 1, stream_boundary)
    launch_box(u, kappa, source, u_next, dt_step, dx, dy,
               ctx.ny, ctx.ny + 1, 1, ctx.nx + 1, stream_boundary)
    launch_box(u, kappa, source, u_next, dt_step, dx, dy,
               2, ctx.ny, 1, 2, stream_boundary)
    launch_box(u, kappa, source, u_next, dt_step, dx, dy,
               2, ctx.ny, ctx.nx, ctx.nx + 1, stream_boundary)
    if events is not None:
        events[3].record(stream_boundary)

    stream_interior.synchronize()
    stream_boundary.synchronize()
    if events is None:
        return None
    return {
        "interior_kernel_ms": float(cuda.event_elapsed_time(events[0], events[1])),
        "boundary_ring_ms": float(cuda.event_elapsed_time(events[2], events[3])),
    }


def distributed_diagnostics(ctx, u, prev, d_e, d_r, dx, dy):
    total_cells = ctx.nx * ctx.ny
    blocks = (total_cells + 255) // 256
    diagnostics_kernel[blocks, 256](u, prev, d_e, d_r, ctx.ny, ctx.nx)
    cuda.synchronize()
    local = np.asarray(
        [np.sum(d_e.copy_to_host()), np.sum(d_r.copy_to_host())],
        dtype=np.float64,
    ) * (dx * dy)
    global_stats = np.empty_like(local)
    ctx.cart.Allreduce(local, global_stats, op=MPI.SUM)
    return float(global_stats[0]), math.sqrt(max(0.0, float(global_stats[1])))


def audit_ptx():
    signatures = tiled_stencil_kernel.signatures
    if not signatures:
        raise RuntimeError("Stencil kernel has not been compiled on a real GPU")
    asm = tiled_stencil_kernel.inspect_asm(signatures[0])
    if isinstance(asm, dict):
        asm = next(iter(asm.values()))
    counts = {
        "st.shared.f64": len(re.findall(r"\bst\.shared\.f64\b", asm)),
        "ld.shared.f64": len(re.findall(r"\bld\.shared\.f64\b", asm)),
        "bar.sync": len(re.findall(r"\bbar\.sync\b", asm)),
        "st.shared.any": len(re.findall(r"\bst\.shared\.", asm)),
        "ld.shared.any": len(re.findall(r"\bld\.shared\.", asm)),
    }
    counts["ptx_audit_passed"] = (
        counts["st.shared.any"] > 0 and
        counts["ld.shared.any"] > 0 and counts["bar.sync"] > 0
    )
    return counts


def run_mode(ctx, u0, kappa_dev, source_dev, dt_step, dx, dy,
             halo, stream_boundary, stream_interior, overlap):
    current = cuda.to_device(u0)
    scratch = cuda.device_array(u0.shape, dtype=np.float64)
    d_e = cuda.device_array((ctx.ny * ctx.nx + 255) // 256, dtype=np.float64)
    d_r = cuda.device_array_like(d_e)
    mode_name = "overlapped" if overlap else "serialized"
    energies = {}
    residuals = {}
    event_samples = []
    energy0, residual0 = distributed_diagnostics(
        ctx, current, current, d_e, d_r, dx, dy
    )
    energies["0"] = energy0
    residuals["0"] = residual0
    if ctx.rank == 0:
        print(f"{mode_name} step=0 energy={energy0:.16e} l2_residual={residual0:.16e}",
              flush=True)
    ctx.cart.Barrier()
    started = MPI.Wtime()
    previous = current
    for t in range(1, STEPS + 1):
        sample = t % 100 == 0
        gpu_events = step_overlapped(
            ctx, current, scratch, kappa_dev, source_dev, dt_step, dx, dy,
            halo, stream_boundary, stream_interior,
            overlap=overlap, sample_events=sample,
        )
        previous, current, scratch = current, scratch, current
        if sample:
            energy, residual = distributed_diagnostics(
                ctx, current, previous, d_e, d_r, dx, dy
            )
            energies[str(t)] = energy
            residuals[str(t)] = residual
            if gpu_events:
                event_samples.append(gpu_events)
            if ctx.rank == 0:
                print(f"{mode_name} step={t} energy={energy:.16e} "
                      f"l2_residual={residual:.16e}", flush=True)
    ctx.cart.Barrier()
    local_ms = 1000.0 * (MPI.Wtime() - started)
    send_time = np.asarray([local_ms], dtype=np.float64)
    global_max_time = np.empty_like(send_time)
    ctx.cart.Allreduce(send_time, global_max_time, op=MPI.MAX)

    # Event samples are per-context and can differ across ranks. Report max.
    if event_samples:
        local_event_means = np.array([
            np.mean([e[k] for e in event_samples])
            for k in ("interior_kernel_ms", "boundary_ring_ms")
        ], dtype=np.float64)
        global_event_means = np.empty_like(local_event_means)
        ctx.cart.Allreduce(local_event_means, global_event_means, op=MPI.MAX)
    else:
        global_event_means = [0.0, 0.0]
    return current, {
        "wall_ms": float(global_max_time[0]),
        "energy_trace": energies,
        "l2_residual_trace": residuals,
        "cuda_event_ms": {
            "interior_kernel": float(global_event_means[0]),
            "boundary_ring": float(global_event_means[1]),
            "sample_count": len(event_samples),
        },
    }


def run_solver(ctx: DistributedContext):
    u0, kappa, source, dx, dy = initialize_subdomains(ctx)
    kappa_local_max = np.array([np.max(kappa[1:-1, 1:-1])], dtype=np.float64)
    kappa_global_max = np.empty_like(kappa_local_max)
    ctx.cart.Allreduce(kappa_local_max, kappa_global_max, op=MPI.MAX)
    kappa_max = float(kappa_global_max[0])
    dt_max = dx * dx / (4.0 * kappa_max)
    dt_step = 0.8 * dt_max
    kappa_dev = cuda.to_device(kappa)
    source_dev = cuda.to_device(source)
    halo = HaloBuffers(ctx.ny, ctx.nx)
    stream_boundary = cuda.stream()
    stream_interior = cuda.stream()
    if ctx.rank == 0:
        print(f"Identity={ctx.github_user} GPU_UUID={ctx.gpu_uuid} "
              f"INSTANCE_SEED={ctx.seed}", flush=True)
        print(f"mpi_ranks={ctx.size} dims=[2,2] local_shape="
              f"({ctx.ny + 2},{ctx.nx + 2}) "
              f"max_kappa={kappa_max:.16e} dx={dx:.16e} "
              f"dt_max={dt_max:.16e} dt={dt_step:.16e}", flush=True)
    print(f"rank={ctx.rank} coords={ctx.coords} neighbors={ctx.neighbors}",
          flush=True)

    # Compile all kernels and start the GPU contexts before comparing modes.
    warm_u = cuda.to_device(u0)
    warm_out = cuda.device_array_like(warm_u)
    step_overlapped(ctx, warm_u, warm_out, kappa_dev, source_dev,
                    dt_step, dx, dy, halo, stream_boundary, stream_interior,
                    overlap=True)
    warm_e = cuda.device_array((ctx.ny * ctx.nx + 255) // 256,
                               dtype=np.float64)
    warm_r = cuda.device_array_like(warm_e)
    distributed_diagnostics(ctx, warm_out, warm_u, warm_e, warm_r, dx, dy)
    ctx.cart.Barrier()

    final_serialized, report_serialized = run_mode(
        ctx, u0, kappa_dev, source_dev, dt_step, dx, dy, halo,
        stream_boundary, stream_interior, overlap=False,
    )
    final_overlap, report_overlap = run_mode(
        ctx, u0, kappa_dev, source_dev, dt_step, dx, dy, halo,
        stream_boundary, stream_interior, overlap=True,
    )

    # Each rank compares ONLY its own region. Allreduce returns one scalar.
    local_diff = float(np.max(np.abs(
        final_serialized.copy_to_host()[1:-1, 1:-1]
        - final_overlap.copy_to_host()[1:-1, 1:-1]
    )))
    local_diff_array = np.array([local_diff], dtype=np.float64)
    global_diff_array = np.empty_like(local_diff_array)
    ctx.cart.Allreduce(local_diff_array, global_diff_array, op=MPI.MAX)
    parity = float(global_diff_array[0])

    # CUDA PTX is architecture and compiler-version dependent. Audit the
    # *actual* JIT output from this run, never hardcode instruction counts.
    ptx = audit_ptx()
    local_ptx_ok = np.array([int(ptx["ptx_audit_passed"])], dtype=np.int32)
    all_ranks_ptx_ok = np.empty_like(local_ptx_ok)
    ctx.cart.Allreduce(local_ptx_ok, all_ranks_ptx_ok, op=MPI.MIN)
    global_ptx_passed = bool(all_ranks_ptx_ok[0])
    if ctx.rank == 0:
        trace = [report_overlap["energy_trace"][str(i)]
                 for i in (0, 100, 200, 300, 400)]
        energy_monotonic = all(
            trace[i + 1] <= trace[i] + 1e-15
            for i in range(len(trace) - 1)
        )
        manifest = {
            "github_username": ctx.github_user,
            "gpu_uuid": ctx.gpu_uuid,
            "instance_seed": ctx.seed,
            "mpi_ranks": 4,
            "cart_dims": [2, 2],
            "cart_periods": [False, False],
            "global_grid": [GLOBAL_NY, GLOBAL_NX],
            "local_padded_shape": [ctx.ny + 2, ctx.nx + 2],
            "steps": STEPS,
            "lambda_damping": LAMBDA_DAMP,
            "max_diffusivity": kappa_max,
            "dx": dx,
            "dy": dy,
            "dt_max": dt_max,
            "dt": dt_step,
            "serialized_ms": report_serialized["wall_ms"],
            "overlapped_ms": report_overlap["wall_ms"],
            "overlap_speedup": report_serialized["wall_ms"] /
                               report_overlap["wall_ms"],
            "parity_max_diff": parity,
            "parity_passed": parity < 1e-12,
            "ptx_audit_passed": global_ptx_passed,
            "ptx_instruction_counts": ptx,
            "cuda_event_ms": {
                "serialized": report_serialized["cuda_event_ms"],
                "overlapped": report_overlap["cuda_event_ms"],
            },
            "energy_trace": {
                "serialized": report_serialized["energy_trace"],
                "overlapped": report_overlap["energy_trace"],
            },
            "l2_residual_trace": {
                "serialized": report_serialized["l2_residual_trace"],
                "overlapped": report_overlap["l2_residual_trace"],
            },
            "energy_monotonic_nonincreasing": energy_monotonic,
            "generated_utc": dt.datetime.now(dt.timezone.utc).isoformat(),
        }
        Path("submission_manifest.json").write_text(
            json.dumps(manifest, indent=2, ensure_ascii=False) + "\n",
            encoding="utf-8",
        )
        print("PTX_AUDIT=" + json.dumps(ptx, sort_keys=True), flush=True)
        print(f"serialized_ms={report_serialized['wall_ms']:.6f} "
              f"overlapped_ms={report_overlap['wall_ms']:.6f} "
              f"speedup={manifest['overlap_speedup']:.6f}", flush=True)
        print(f"parity_max_diff={parity:.16e} "
              f"parity_passed={manifest['parity_passed']} "
              f"energy_monotonic={energy_monotonic}", flush=True)
        print("Wrote submission_manifest.json", flush=True)
    ctx.cart.Barrier()
    if not global_ptx_passed or parity >= 1e-12:
        raise RuntimeError("Parity or PTX audit did not pass. Inspect manifest.")


def main():
    if len(sys.argv) != 2:
        raise SystemExit("Usage: python3 distributed_cuda_solver.py GITHUB_USERNAME")
    ctx = init_distributed_context(sys.argv[1])
    run_solver(ctx)


if __name__ == "__main__":
    main()


Writing distributed_cuda_solver.py


## Analysis helper — Generate answers using only real run telemetry

In [4]:
%%writefile answers_generator.py
"""Build six *run-grounded* answers from the real CUDA/MPI JSON manifest.

No PTX instruction count, GPU timing, energy, or parity value is invented.
"""

from __future__ import annotations

import json
from pathlib import Path


def load_manifest(path="submission_manifest.json"):
    return json.loads(Path(path).read_text(encoding="utf-8"))


def make_answers(m):
    shape = m["local_padded_shape"]
    n = shape[0] - 2
    pitch = shape[1]
    per_column_span = ((n - 1) * pitch + 1) * 8
    useful = n * 8
    extra = per_column_span - useful
    full_rectangle = n * pitch * 8
    full_rectangle_extra = full_rectangle - useful
    active_column_messages = 4  # each of four ranks in a 2x2 grid has one x-neighbour
    counts = m["ptx_instruction_counts"]
    wall_ser = m["serialized_ms"]
    wall_ov = m["overlapped_ms"]
    speed = m["overlap_speedup"]
    times = m["cuda_event_ms"]
    trace = m["energy_trace"]["overlapped"]
    residual = m["l2_residual_trace"]["overlapped"]
    values = [(k, trace[str(k)], residual[str(k)])
              for k in (0, 100, 200, 300, 400)]
    trace_table = "\n".join(
        f"| {step} | {e:.16e} | {r:.16e} |"
        for step, e, r in values
    )
    differences = [values[i + 1][1] - values[i][1] for i in range(4)]
    change_text = ", ".join(f"{x:+.6e}" for x in differences)
    monotonic = m["energy_monotonic_nonincreasing"]
    conclusion = (
        "All four measured intervals are non-increasing, consistent with a "
        "dissipative solution for these initial conditions and this small source."
        if monotonic else
        "The measured energy is NOT monotonically non-increasing. The source "
        "can inject heat; therefore a monotonicity claim would be incorrect."
    )
    gain = "faster" if speed > 1 else "slower"

    answers = {}
    answers["Q1"] = f"""### Q1 — Memory coalescing and GPU column packing

The field is C-contiguous `float64` with `{shape[1]}` elements per row (pitch =
`{pitch * 8}` bytes). A halo column has `{n}` values, but consecutive column
values are `{pitch * 8}` bytes apart. A single linear DMA cannot read only those
separated elements. CUDA packing makes a new **contiguous** `{n}`-element vector
(`{useful:,}` useful bytes) before copying through PCIe. A warp packing a column
has strided input loads; the output stores into consecutive vector positions.

If one tried a single contiguous DMA covering the first to last column entry,
its span would be `(({n}-1)*{pitch}+1)*8 = {per_column_span:,}` bytes. That is
**{extra:,} extra bytes per column**, versus `{useful:,}` useful bytes. Copying
both the left and right columns naively would waste `{2 * extra:,}` bytes per
rank and step. In our nonperiodic 2×2 topology each rank communicates one
horizontal column, so over all four ranks the potential wasted span is
`4 × {extra:,} = {active_column_messages * extra:,}` extra bytes per timestep,
or `{active_column_messages * extra * m['steps']:,}` bytes across {m['steps']}
steps. For a larger **whole-row rectangle** (all {n} rows by {pitch} columns), the copy
would instead be `{full_rectangle:,}` bytes, or `{full_rectangle_extra:,}`
extra per column. These count payload spans only, not PCIe transaction
overhead.
"""
    answers["Q2"] = """### Q2 — Shared-memory indexing and bank conflicts

Each thread `(tx, ty)` writes `smem[ty+1, tx+1]` into an 18×18 `float64` tile.
Thread `(0,0)` writes the center at **`[1,1]`** and loads the west apron
**`[1,0]`** and north apron **`[0,1]`**. It later reads its center `[1,1]`,
east `[1,2]`, west `[1,0]`, south `[2,1]`, and north `[0,1]`.
Thread `(15,15)` writes **`[16,16]`**, loads east **`[16,17]`** and south
**`[17,16]`**, then reads `[16,16]`, `[16,17]`, `[16,15]`, `[17,16]`,
and `[15,16]`. No diagonal corners are needed for the five-point stencil.

On the T4, 32 banks are 4 bytes wide; a `float64` occupies two adjacent bank
words. Along a row, adjacent 8-byte values use adjacent pairs of banks.
Sixteen contiguous double loads fit 128 bytes and can be serviced without an
intrinsic 2-way conflict: a larger warp operation splits into transactions.
For a **hypothetical column-wise set of threads**, the 18-double row stride
is `18×8=144` bytes = 36 bank words, giving bank-index increment
`36 mod 32 = 4`. The starting bank repeats after 8 rows: 16 simultaneous
strided lanes may have a 2-way bank-alias pattern, and 32 lanes a 4-way
pattern, before transaction splitting. Merely using `float64` does **not**
mean every access is a 2-way conflict. In the actual 16×16 threadblock a
32-lane warp spans two adjacent 16-element **rows**, and each half-warp reads
contiguous columns, so it does not execute the hypothetical one-lane-per-row
column pattern.
"""
    answers["Q3"] = f"""### Q3 — PTX audit and cross-warp synchronization

The **actual** `inspect_asm()` counts from this run were:

| PTX instruction | Count |
|---|---:|
| `st.shared.f64` | {counts['st.shared.f64']} |
| `ld.shared.f64` | {counts['ld.shared.f64']} |
| `bar.sync` | {counts['bar.sync']} |
| all `st.shared.*` variants | {counts['st.shared.any']} |
| all `ld.shared.*` variants | {counts['ld.shared.any']} |

The audit result was **`{str(m['ptx_audit_passed']).lower()}`**. PTX counts are
specific to the generated kernel, numba/NVVM toolchain and T4, not constants
that should be copied from another computer. The 16×16 block has eight
32-thread warps. At a warp boundary a thread's stencil may consume apron or
center values written by **another warp**. Without the block-level
`cuda.syncthreads()` / `bar.sync`, a reading warp could run ahead before its
neighbor has stored the tile, producing a read-after-write race with
uninitialized/stale shared-memory contents. Warp-local lockstep is not a
substitute for cross-warp synchronization.
"""
    answers["Q4"] = f"""### Q4 — Four processes and one Tesla T4

All four `mpirun -np 4` ranks create **separate CUDA contexts on GPU 0**. SMs,
L2 and copy engines are a *shared* resource, not four independent accelerators.
Without an explicitly enabled multi-process service (MPS), context scheduling
and time-slicing can further limit concurrent SM execution. Each process's
host-staged `cuda.pinned_array` DMA transfers and its MPI communication also
compete for the same underlying GPU/host resources.

The measured global-max elapsed times were **serialized = {wall_ser:.3f} ms**
and **overlapped = {wall_ov:.3f} ms**; speedup
`serialized_ms / overlapped_ms = {speed:.4f}×`. On this run, the overlapped
pipeline was **{gain}**. Recorded CUDA event averages (maximum across ranks)
were:

| Mode | Interior kernel (ms) | Four boundary-ring kernels (ms) |
|---|---:|---:|
| Serialized | {times['serialized']['interior_kernel']:.6f} | {times['serialized']['boundary_ring']:.6f} |
| Overlapped | {times['overlapped']['interior_kernel']:.6f} | {times['overlapped']['boundary_ring']:.6f} |

The event values are GPU-kernel durations, **not** the full MPI or PCIe
critical path. With four dedicated GPUs connected by InfiniBand, each rank
would have independent SM resources and the opportunity to hide network
latency under local interior computation. Four processes sharing one T4 do
not have that same independent execution capacity; contention can negate
any overlap benefit.
"""
    answers["Q5"] = """### Q5 — MPI blocking deadlock and PROC_NULL

If every rank performs `Send` before its matching `Recv`, messages below the
MPI *eager* threshold may appear to work: the MPI library internally buffers
a small send and returns before the peer posts a receive. For larger halo
messages (or exhausted eager buffers), MPI typically switches to a
**rendezvous** protocol. A blocking `Send` then waits for receiver handshake
or completion. If neighboring ranks are all waiting in sends that require
receives not yet posted, the wait-for graph contains a cycle: **deadlock**.
An implementation must not depend on an eager threshold, since it varies
between MPI stacks and transports. Posting tag-matched `Irecv` operations
first and then `Isend`, followed by `MPI.Request.Waitall`, removes this
Send-first dependency and allows the interior computation to proceed while
the exchange is pending.

For corner Cartesian coordinates `(0,0)`, `Shift(0,1)` gives no up neighbor
and `Shift(1,1)` gives no left neighbor: both are `MPI.PROC_NULL`. MPI
send/receive operations involving PROC_NULL are no-ops (a receive does not
populate a data buffer). We therefore *explicitly set* the corresponding
ghost cells to zero to implement the required homogeneous Dirichlet
boundary. The down and right neighbors exchange normal messages.
"""
    answers["Q6"] = f"""### Q6 — Explicit CFL limit and measured dissipative behavior

This run measured `kappa_max = {m['max_diffusivity']:.16e}` and
`dx = dy = 1/513 = {m['dx']:.16e}`. The PDF's conservative two-dimensional
explicit limit is

`dt_max = dx² / (4*kappa_max) = {m['dt_max']:.16e}`.

The implemented timestep was `dt = {m['dt']:.16e}` =
`{m['dt']/m['dt_max']:.4f} * dt_max`, hence **below the stated CFL limit**.
The actual MPI `Allreduce(SUM)` totals below were computed over each rank's
*owned interior only*, never by gathering a global field:

| Step | E = sum(u)·dx·dy | Global L2 change residual |
|---:|---:|---:|
{trace_table}

Successive energy changes are `{change_text}`. {conclusion}
The measured local-vs-global comparison also gives final
**`parity_max_diff = {m['parity_max_diff']:.16e}`**
(`< 1e-12`: **{str(m['parity_passed']).lower()}**).

**Important mathematical distinction:** `sum(u)*dx*dy` is not a generally
conserved invariant here, because the PDE explicitly contains the negative
cubic reaction term and a positive localized source, in addition to flux
through zero-valued physical boundaries. The trace demonstrates the observed
behavior for this run; satisfying the diffusion CFL bound alone does not
prove monotonic energy decay in the presence of arbitrary forcing.
"""
    return answers


def markdown_all(m):
    answers = make_answers(m)
    return "# Analytical & Forensic Engineering Responses\n\n" + "\n\n".join(
        answers[f"Q{k}"] for k in range(1, 7)
    ) + "\n"


def main():
    manifest = load_manifest()
    md = markdown_all(manifest)
    Path("answers_generated.md").write_text(md, encoding="utf-8")
    print("Wrote answers_generated.md from real submission_manifest.json")
    print(md)


if __name__ == "__main__":
    main()


Writing answers_generator.py


## Finalization helper — Replace six Markdown cells without altering execution stdout

In [5]:
%%writefile finalize_notebook.py
"""Fill six Markdown answer cells in a REAL executed exported notebook.

Usage:
    python finalize_notebook.py mpi_cuda_stencil_lab.ipynb submission_manifest.json

Output: mpi_cuda_stencil_lab_completed.ipynb; original code execution outputs
are not modified. Do not use this on an unexecuted Colab notebook.
"""

from pathlib import Path
import sys
import nbformat
from answers_generator import load_manifest, make_answers


def finalize(infile, manifestfile, outfile):
    nb = nbformat.read(str(infile), as_version=4)
    answers = make_answers(load_manifest(manifestfile))
    replaced = set()
    for cell in nb.cells:
        if cell.cell_type != "markdown":
            continue
        for key, answer in answers.items():
            if cell.source.startswith(f"<!-- AUTO:FORENSIC:{key} -->"):
                cell.source = f"<!-- AUTO:FORENSIC:{key} -->\n\n{answer}"
                replaced.add(key)
                break
    if replaced != set(answers):
        raise RuntimeError(f"Missing answer cell markers: {set(answers) - replaced}")
    if not any(cell.cell_type == "code" and cell.get("outputs")
               and "mpirun" in cell.source for cell in nb.cells):
        raise RuntimeError("Notebook lacks execution output for MPI launcher")
    nbformat.write(nb, str(outfile))
    print(f"Saved {outfile}; replaced all 6 Markdown cells; kept original stdout.")


if __name__ == "__main__":
    if len(sys.argv) not in (3, 4):
        raise SystemExit("Usage: python finalize_notebook.py notebook.ipynb manifest.json [output.ipynb]")
    target = Path(sys.argv[3]) if len(sys.argv) == 4 else Path("mpi_cuda_stencil_lab_completed.ipynb")
    finalize(Path(sys.argv[1]), Path(sys.argv[2]), target)


Writing finalize_notebook.py


## Cell 4 — Four-rank CUDA execution (both serialized and overlapped, 400 steps each)

In [6]:
import subprocess
from pathlib import Path
cmd = [
    "mpirun", "--allow-run-as-root", "--oversubscribe",
    "-np", "4", "python3", "distributed_cuda_solver.py", GITHUB_USERNAME,
]
proc = subprocess.run(cmd, capture_output=True, text=True)
print(proc.stdout, end="")  # unedited stdout from all MPI ranks
if proc.returncode != 0:
    print("STDERR:\n" + proc.stderr)
    raise RuntimeError(f"MPI failed with exit status {proc.returncode}")
assert Path("submission_manifest.json").is_file()
print("Real GPU run complete; submission_manifest.json exists")

rank=2 coords=(1, 0) neighbors=(0, -2, -2, 3)
rank=3 coords=(1, 1) neighbors=(1, -2, 2, -2)
Identity=allfocusSS GPU_UUID=GPU-561dd3dd-1373-c02a-71f0-8c2c965c50bc INSTANCE_SEED=15290349302218891270
mpi_ranks=4 dims=[2,2] local_shape=(258,258) max_kappa=2.3999249955313717e-01 dx=1.9493177387914229e-03 dt_max=3.9582900026481621e-06 dt=3.1666320021185297e-06
rank=0 coords=(0, 0) neighbors=(-2, 2, -2, 1)
rank=1 coords=(0, 1) neighbors=(-2, 3, 0, -2)
serialized step=0 energy=8.3941786911102642e-02 l2_residual=0.0000000000000000e+00
serialized step=100 energy=8.3928497550886785e-02 l2_residual=1.6845821308125397e-05
serialized step=200 energy=8.3919131871226391e-02 l2_residual=1.6538193507168178e-05
serialized step=300 energy=8.3910483903346614e-02 l2_residual=1.6247531466674538e-05
serialized step=400 energy=8.3902138161996515e-02 l2_residual=1.5969433852180116e-05
overlapped step=0 energy=8.3941786911102642e-02 l2_residual=0.0000000000000000e+00
overlapped step=100 energy=8.3928497550886785

## Dedicated forensic Markdown cells (completed from the measured run)

These six Markdown cells are replaced automatically by `finalize_notebook.py` after exporting the **executed** notebook. The rendered answers in Cell 5 already display all six full explanations and numerical values.

### Q1 — Memory coalescing and GPU column packing

The field is C-contiguous `float64` with `258` elements per row (pitch =
`2064` bytes). A halo column has `256` values, but consecutive column
values are `2064` bytes apart. A single linear DMA cannot read only those
separated elements. CUDA packing makes a new **contiguous** `256`-element vector
(`2,048` useful bytes) before copying through PCIe. A warp packing a column
has strided input loads; the output stores into consecutive vector positions.

If one tried a single contiguous DMA covering the first to last column entry,
its span would be `((256-1)*258+1)*8 = 526,328` bytes. That is
**524,280 extra bytes per column**, versus `2,048` useful bytes. Copying
both the left and right columns naively would waste `1,048,560` bytes per
rank and step. In our nonperiodic 2×2 topology each rank communicates one
horizontal column, so over all four ranks the potential wasted span is
`4 × 524,280 = 2,097,120` extra bytes per timestep,
or `838,848,000` bytes across 400
steps. For a larger **whole-row rectangle** (all 256 rows by 258 columns), the copy
would instead be `528,384` bytes, or `526,336`
extra per column. These count payload spans only, not PCIe transaction
overhead.


### Q2 — Shared-memory indexing and bank conflicts

Each thread `(tx, ty)` writes `smem[ty+1, tx+1]` into an 18×18 `float64` tile.
Thread `(0,0)` writes the center at **`[1,1]`** and loads the west apron
**`[1,0]`** and north apron **`[0,1]`**. It later reads its center `[1,1]`,
east `[1,2]`, west `[1,0]`, south `[2,1]`, and north `[0,1]`.
Thread `(15,15)` writes **`[16,16]`**, loads east **`[16,17]`** and south
**`[17,16]`**, then reads `[16,16]`, `[16,17]`, `[16,15]`, `[17,16]`,
and `[15,16]`. No diagonal corners are needed for the five-point stencil.

On the T4, 32 banks are 4 bytes wide; a `float64` occupies two adjacent bank
words. Along a row, adjacent 8-byte values use adjacent pairs of banks.
Sixteen contiguous double loads fit 128 bytes and can be serviced without an
intrinsic 2-way conflict: a larger warp operation splits into transactions.
For a **hypothetical column-wise set of threads**, the 18-double row stride
is `18×8=144` bytes = 36 bank words, giving bank-index increment
`36 mod 32 = 4`. The starting bank repeats after 8 rows: 16 simultaneous
strided lanes may have a 2-way bank-alias pattern, and 32 lanes a 4-way
pattern, before transaction splitting. Merely using `float64` does **not**
mean every access is a 2-way conflict. In the actual 16×16 threadblock a
32-lane warp spans two adjacent 16-element **rows**, and each half-warp reads
contiguous columns, so it does not execute the hypothetical one-lane-per-row
column pattern.


### Q3 — PTX audit and cross-warp synchronization

The **actual** `inspect_asm()` counts from this run were:

| PTX instruction | Count |
|---|---:|
| `st.shared.f64` | 5 |
| `ld.shared.f64` | 5 |
| `bar.sync` | 1 |
| all `st.shared.*` variants | 5 |
| all `ld.shared.*` variants | 5 |

The audit result was **`true`**. PTX counts are
specific to the generated kernel, numba/NVVM toolchain and T4, not constants
that should be copied from another computer. The 16×16 block has eight
32-thread warps. At a warp boundary a thread's stencil may consume apron or
center values written by **another warp**. Without the block-level
`cuda.syncthreads()` / `bar.sync`, a reading warp could run ahead before its
neighbor has stored the tile, producing a read-after-write race with
uninitialized/stale shared-memory contents. Warp-local lockstep is not a
substitute for cross-warp synchronization.


### Q4 — Four processes and one Tesla T4

All four `mpirun -np 4` ranks create **separate CUDA contexts on GPU 0**. SMs,
L2 and copy engines are a *shared* resource, not four independent accelerators.
Without an explicitly enabled multi-process service (MPS), context scheduling
and time-slicing can further limit concurrent SM execution. Each process's
host-staged `cuda.pinned_array` DMA transfers and its MPI communication also
compete for the same underlying GPU/host resources.

The measured global-max elapsed times were **serialized = 1909.157 ms**
and **overlapped = 1568.603 ms**; speedup
`serialized_ms / overlapped_ms = 1.2171×`. On this run, the overlapped
pipeline was **faster**. Recorded CUDA event averages (maximum across ranks)
were:

| Mode | Interior kernel (ms) | Four boundary-ring kernels (ms) |
|---|---:|---:|
| Serialized | 0.252080 | 0.947912 |
| Overlapped | 1.242608 | 1.003936 |

The event values are GPU-kernel durations, **not** the full MPI or PCIe
critical path. With four dedicated GPUs connected by InfiniBand, each rank
would have independent SM resources and the opportunity to hide network
latency under local interior computation. Four processes sharing one T4 do
not have that same independent execution capacity; contention can negate
any overlap benefit.


### Q5 — MPI blocking deadlock and PROC_NULL

If every rank performs `Send` before its matching `Recv`, messages below the
MPI *eager* threshold may appear to work: the MPI library internally buffers
a small send and returns before the peer posts a receive. For larger halo
messages (or exhausted eager buffers), MPI typically switches to a
**rendezvous** protocol. A blocking `Send` then waits for receiver handshake
or completion. If neighboring ranks are all waiting in sends that require
receives not yet posted, the wait-for graph contains a cycle: **deadlock**.
An implementation must not depend on an eager threshold, since it varies
between MPI stacks and transports. Posting tag-matched `Irecv` operations
first and then `Isend`, followed by `MPI.Request.Waitall`, removes this
Send-first dependency and allows the interior computation to proceed while
the exchange is pending.

For corner Cartesian coordinates `(0,0)`, `Shift(0,1)` gives no up neighbor
and `Shift(1,1)` gives no left neighbor: both are `MPI.PROC_NULL`. MPI
send/receive operations involving PROC_NULL are no-ops (a receive does not
populate a data buffer). We therefore *explicitly set* the corresponding
ghost cells to zero to implement the required homogeneous Dirichlet
boundary. The down and right neighbors exchange normal messages.


### Q6 — Explicit CFL limit and measured dissipative behavior

This run measured `kappa_max = 2.3999249955313717e-01` and
`dx = dy = 1/513 = 1.9493177387914229e-03`. The PDF's conservative two-dimensional
explicit limit is

`dt_max = dx² / (4*kappa_max) = 3.9582900026481621e-06`.

The implemented timestep was `dt = 3.1666320021185297e-06` =
`0.8000 * dt_max`, hence **below the stated CFL limit**.
The actual MPI `Allreduce(SUM)` totals below were computed over each rank's
*owned interior only*, never by gathering a global field:

| Step | E = sum(u)·dx·dy | Global L2 change residual |
|---:|---:|---:|
| 0 | 8.3941786911102642e-02 | 0.0000000000000000e+00 |
| 100 | 8.3928497550886785e-02 | 1.6845821308125397e-05 |
| 200 | 8.3919131871226391e-02 | 1.6538193507168178e-05 |
| 300 | 8.3910483903346614e-02 | 1.6247531466674538e-05 |
| 400 | 8.3902138161996515e-02 | 1.5969433852180116e-05 |

Successive energy changes are `-1.328936e-05, -9.365680e-06, -8.647968e-06, -8.345741e-06`. All four measured intervals are non-increasing, consistent with a dissipative solution for these initial conditions and this small source.
The measured local-vs-global comparison also gives final
**`parity_max_diff = 0.0000000000000000e+00`**
(`< 1e-12`: **true**).

**Important mathematical distinction:** `sum(u)*dx*dy` is not a generally
conserved invariant here, because the PDE explicitly contains the negative
cubic reaction term and a positive localized source, in addition to flux
through zero-valued physical boundaries. The trace demonstrates the observed
behavior for this run; satisfying the diffusion CFL bound alone does not
prove monotonic energy decay in the presence of arbitrary forcing.


## Cell 5 — Render actual quantitative Q1–Q6 answers

In [7]:
from IPython.display import display, Markdown
from answers_generator import load_manifest, markdown_all
manifest = load_manifest("submission_manifest.json")
answers_md = markdown_all(manifest)
Path("answers_generated.md").write_text(answers_md, encoding="utf-8")
display(Markdown(answers_md))
print("Rendered six answers using measured manifest; saved answers_generated.md")

# Analytical & Forensic Engineering Responses

### Q1 — Memory coalescing and GPU column packing

The field is C-contiguous `float64` with `258` elements per row (pitch =
`2064` bytes). A halo column has `256` values, but consecutive column
values are `2064` bytes apart. A single linear DMA cannot read only those
separated elements. CUDA packing makes a new **contiguous** `256`-element vector
(`2,048` useful bytes) before copying through PCIe. A warp packing a column
has strided input loads; the output stores into consecutive vector positions.

If one tried a single contiguous DMA covering the first to last column entry,
its span would be `((256-1)*258+1)*8 = 526,328` bytes. That is
**524,280 extra bytes per column**, versus `2,048` useful bytes. Copying
both the left and right columns naively would waste `1,048,560` bytes per
rank and step. In our nonperiodic 2×2 topology each rank communicates one
horizontal column, so over all four ranks the potential wasted span is
`4 × 524,280 = 2,097,120` extra bytes per timestep,
or `838,848,000` bytes across 400
steps. For a larger **whole-row rectangle** (all 256 rows by 258 columns), the copy
would instead be `528,384` bytes, or `526,336`
extra per column. These count payload spans only, not PCIe transaction
overhead.


### Q2 — Shared-memory indexing and bank conflicts

Each thread `(tx, ty)` writes `smem[ty+1, tx+1]` into an 18×18 `float64` tile.
Thread `(0,0)` writes the center at **`[1,1]`** and loads the west apron
**`[1,0]`** and north apron **`[0,1]`**. It later reads its center `[1,1]`,
east `[1,2]`, west `[1,0]`, south `[2,1]`, and north `[0,1]`.
Thread `(15,15)` writes **`[16,16]`**, loads east **`[16,17]`** and south
**`[17,16]`**, then reads `[16,16]`, `[16,17]`, `[16,15]`, `[17,16]`,
and `[15,16]`. No diagonal corners are needed for the five-point stencil.

On the T4, 32 banks are 4 bytes wide; a `float64` occupies two adjacent bank
words. Along a row, adjacent 8-byte values use adjacent pairs of banks.
Sixteen contiguous double loads fit 128 bytes and can be serviced without an
intrinsic 2-way conflict: a larger warp operation splits into transactions.
For a **hypothetical column-wise set of threads**, the 18-double row stride
is `18×8=144` bytes = 36 bank words, giving bank-index increment
`36 mod 32 = 4`. The starting bank repeats after 8 rows: 16 simultaneous
strided lanes may have a 2-way bank-alias pattern, and 32 lanes a 4-way
pattern, before transaction splitting. Merely using `float64` does **not**
mean every access is a 2-way conflict. In the actual 16×16 threadblock a
32-lane warp spans two adjacent 16-element **rows**, and each half-warp reads
contiguous columns, so it does not execute the hypothetical one-lane-per-row
column pattern.


### Q3 — PTX audit and cross-warp synchronization

The **actual** `inspect_asm()` counts from this run were:

| PTX instruction | Count |
|---|---:|
| `st.shared.f64` | 5 |
| `ld.shared.f64` | 5 |
| `bar.sync` | 1 |
| all `st.shared.*` variants | 5 |
| all `ld.shared.*` variants | 5 |

The audit result was **`true`**. PTX counts are
specific to the generated kernel, numba/NVVM toolchain and T4, not constants
that should be copied from another computer. The 16×16 block has eight
32-thread warps. At a warp boundary a thread's stencil may consume apron or
center values written by **another warp**. Without the block-level
`cuda.syncthreads()` / `bar.sync`, a reading warp could run ahead before its
neighbor has stored the tile, producing a read-after-write race with
uninitialized/stale shared-memory contents. Warp-local lockstep is not a
substitute for cross-warp synchronization.


### Q4 — Four processes and one Tesla T4

All four `mpirun -np 4` ranks create **separate CUDA contexts on GPU 0**. SMs,
L2 and copy engines are a *shared* resource, not four independent accelerators.
Without an explicitly enabled multi-process service (MPS), context scheduling
and time-slicing can further limit concurrent SM execution. Each process's
host-staged `cuda.pinned_array` DMA transfers and its MPI communication also
compete for the same underlying GPU/host resources.

The measured global-max elapsed times were **serialized = 1909.157 ms**
and **overlapped = 1568.603 ms**; speedup
`serialized_ms / overlapped_ms = 1.2171×`. On this run, the overlapped
pipeline was **faster**. Recorded CUDA event averages (maximum across ranks)
were:

| Mode | Interior kernel (ms) | Four boundary-ring kernels (ms) |
|---|---:|---:|
| Serialized | 0.252080 | 0.947912 |
| Overlapped | 1.242608 | 1.003936 |

The event values are GPU-kernel durations, **not** the full MPI or PCIe
critical path. With four dedicated GPUs connected by InfiniBand, each rank
would have independent SM resources and the opportunity to hide network
latency under local interior computation. Four processes sharing one T4 do
not have that same independent execution capacity; contention can negate
any overlap benefit.


### Q5 — MPI blocking deadlock and PROC_NULL

If every rank performs `Send` before its matching `Recv`, messages below the
MPI *eager* threshold may appear to work: the MPI library internally buffers
a small send and returns before the peer posts a receive. For larger halo
messages (or exhausted eager buffers), MPI typically switches to a
**rendezvous** protocol. A blocking `Send` then waits for receiver handshake
or completion. If neighboring ranks are all waiting in sends that require
receives not yet posted, the wait-for graph contains a cycle: **deadlock**.
An implementation must not depend on an eager threshold, since it varies
between MPI stacks and transports. Posting tag-matched `Irecv` operations
first and then `Isend`, followed by `MPI.Request.Waitall`, removes this
Send-first dependency and allows the interior computation to proceed while
the exchange is pending.

For corner Cartesian coordinates `(0,0)`, `Shift(0,1)` gives no up neighbor
and `Shift(1,1)` gives no left neighbor: both are `MPI.PROC_NULL`. MPI
send/receive operations involving PROC_NULL are no-ops (a receive does not
populate a data buffer). We therefore *explicitly set* the corresponding
ghost cells to zero to implement the required homogeneous Dirichlet
boundary. The down and right neighbors exchange normal messages.


### Q6 — Explicit CFL limit and measured dissipative behavior

This run measured `kappa_max = 2.3999249955313717e-01` and
`dx = dy = 1/513 = 1.9493177387914229e-03`. The PDF's conservative two-dimensional
explicit limit is

`dt_max = dx² / (4*kappa_max) = 3.9582900026481621e-06`.

The implemented timestep was `dt = 3.1666320021185297e-06` =
`0.8000 * dt_max`, hence **below the stated CFL limit**.
The actual MPI `Allreduce(SUM)` totals below were computed over each rank's
*owned interior only*, never by gathering a global field:

| Step | E = sum(u)·dx·dy | Global L2 change residual |
|---:|---:|---:|
| 0 | 8.3941786911102642e-02 | 0.0000000000000000e+00 |
| 100 | 8.3928497550886785e-02 | 1.6845821308125397e-05 |
| 200 | 8.3919131871226391e-02 | 1.6538193507168178e-05 |
| 300 | 8.3910483903346614e-02 | 1.6247531466674538e-05 |
| 400 | 8.3902138161996515e-02 | 1.5969433852180116e-05 |

Successive energy changes are `-1.328936e-05, -9.365680e-06, -8.647968e-06, -8.345741e-06`. All four measured intervals are non-increasing, consistent with a dissipative solution for these initial conditions and this small source.
The measured local-vs-global comparison also gives final
**`parity_max_diff = 0.0000000000000000e+00`**
(`< 1e-12`: **true**).

**Important mathematical distinction:** `sum(u)*dx*dy` is not a generally
conserved invariant here, because the PDE explicitly contains the negative
cubic reaction term and a positive localized source, in addition to flux
through zero-valued physical boundaries. The trace demonstrates the observed
behavior for this run; satisfying the diffusion CFL bound alone does not
prove monotonic energy decay in the presence of arbitrary forcing.



Rendered six answers using measured manifest; saved answers_generated.md


## Submission verification (computed from this run)

Verify `mpi_ranks == 4`, `ptx_audit_passed == True`, `parity_max_diff < 1e-12` and the answers contain real numerical data. Do not invent output if your GPU runtime is unavailable.

In [8]:
print("GitHub:", manifest["github_username"])
print("GPU UUID:", manifest["gpu_uuid"])
print("MPI ranks:", manifest["mpi_ranks"])
print("PTX audit passed:", manifest["ptx_audit_passed"])
print("Parity max diff:", manifest["parity_max_diff"])
print("Serialized ms:", manifest["serialized_ms"])
print("Overlapped ms:", manifest["overlapped_ms"])
assert manifest["github_username"] == GITHUB_USERNAME
assert manifest["mpi_ranks"] == 4
assert manifest["ptx_audit_passed"]
assert manifest["parity_max_diff"] < 1e-12

GitHub: allfocusSS
GPU UUID: GPU-561dd3dd-1373-c02a-71f0-8c2c965c50bc
MPI ranks: 4
PTX audit passed: True
Parity max diff: 0.0
Serialized ms: 1909.156977
Overlapped ms: 1568.6031890000002


In [9]:
# Optional: if this notebook file is physically present under /content,
# this can generate an answered copy immediately. Colab often saves the
# notebook only in its own UI, so in that case use the README export route.
from finalize_notebook import finalize
if Path("mpi_cuda_stencil_lab.ipynb").is_file():
    finalize(
        Path("mpi_cuda_stencil_lab.ipynb"),
        Path("submission_manifest.json"),
        Path("mpi_cuda_stencil_lab_completed.ipynb"),
    )
    print("Ready: mpi_cuda_stencil_lab_completed.ipynb")
else:
    print("Next: File -> Download -> Download .ipynb, then run finalizer")

Next: File -> Download -> Download .ipynb, then run finalizer
